# Random Forest – Klasifikace onemocnění srdce (Heart Disease)
Tento Jupyter sešit představuje kompletní vypracované řešení cvičení **Random Forest - classification** ze 3. dne kurzu Machine Learning.

### Zadání cvičení:
1. Načíst data `heart_data_normalized.csv` do proměnné pomocí knihovny Pandas.
2. Provést kontrolu dat a oddělit příznaky $X$ a cílovou proměnnou $y$ (`ahd_yes`).
3. Rozdělit dataset na trénovací a testovací sadu v poměru 70:30 se seedem `random_state=42`.
4. Vytvořit instanci modelu `RandomForestClassifier` s parametry `random_state=42` a `n_jobs=-1`.
5. Definovat slovník hyperparametrů `params` pro klíče `max_depth`, `min_samples_leaf` a `n_estimators` (2 až 4 hodnoty pro každý klíč).
6. Vytvořit objekt `GridSearchCV` pro optimalizaci s metrikou `scoring='precision'`.
7. Natrénovat model na trénovacích datech `X_train, y_train` s nejlepší sadou hyperparametrů.
8. Provést predikci na testovací sadě `X_test` a vygenerovat podrobný `classification_report` a matici záměn.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (9, 5)
print('Knihovny úspěšně importovány.')

Knihovny úspěšně importovány.


In [2]:
# 1. & 2. Načtení dat a kontrola
data_path = '../data/MAL_downloadable materials_session 2/Day 3/heart_data_normalized.csv'
if not os.path.exists(data_path):
    data_path = 'heart_data_normalized.csv'

heart_data = pd.read_csv(data_path)
print('Rozměry datasetu:', heart_data.shape)
print('Rozdělení cílové proměnné (ahd_yes):')
print(heart_data['ahd_yes'].value_counts())
heart_data.head()

Rozměry datasetu: (303, 18)
Rozdělení cílové proměnné (ahd_yes):
0    164
1    139
Name: ahd_yes, dtype: int64


In [3]:
# 3. Příprava matice X a vektoru y + Split 70:30
X = heart_data.drop('ahd_yes', axis=1)
y = heart_data['ahd_yes']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f'Trénovací sada: {X_train.shape}, Testovací sada: {X_test.shape}')

Trénovací sada: (212, 17), Testovací sada: (91, 17)


In [4]:
# 4., 5., 6. & 7. RandomForestClassifier + Definice mřížky + GridSearchCV (scoring='precision')
rf_classifier = RandomForestClassifier(random_state=42, n_jobs=-1)

params = {
    'max_depth': [3, 5, 7, None],
    'min_samples_leaf': [1, 2, 4],
    'n_estimators': [50, 100, 150]
}
print(f"Grid mřížka obsahuje {len(params['max_depth']) * len(params['min_samples_leaf']) * len(params['n_estimators'])} kombinací hyperparametrů.")

grid_search = GridSearchCV(estimator=rf_classifier, param_grid=params, scoring='precision', cv=5, n_jobs=-1)
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
print('Trénování GridSearchCV dokončeno.')
print('Optimální hyperparametry:', grid_search.best_params_)
print(f"Nejvyšší průměrná Precision při křížové validaci (CV=5): {grid_search.best_score_*100:.2f} %")

Grid mřížka obsahuje 36 kombinací hyperparametrů.
Trénování GridSearchCV dokončeno.
Optimální hyperparametry: {'max_depth': 3, 'min_samples_leaf': 2, 'n_estimators': 50}
Nejvyšší průměrná Precision při křížové validaci (CV=5): 85.68 %


In [5]:
# 8. & 9. Predikce na testovací sadě a vygenerování classification_report
y_pred = best_rf.predict(X_test)

print('=== CLASSIFICATION REPORT (Testovací sada 30 %) ===')
print(classification_report(y_test, y_pred))

print('Matice záměn (Confusion Matrix):')
cm = confusion_matrix(y_test, y_pred)
print(cm)

=== CLASSIFICATION REPORT (Testovací sada 30 %) ===
              precision    recall  f1-score   support

           0       0.86      0.88      0.87        48
           1       0.86      0.84      0.85        43

    accuracy                           0.86        91
   macro avg       0.86      0.86      0.86        91
weighted avg       0.86      0.86      0.86        91

Matice záměn (Confusion Matrix):
[[42  6]
 [ 7 36]]


In [6]:
# Vizualizace 1: Matice záměn (Confusion Matrix)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Zdravý (0)', 'Nemoc srdce (1)'],
            yticklabels=['Zdravý (0)', 'Nemoc srdce (1)'])
plt.title('Matice záměn: Random Forest na testovací sadě', fontsize=13)
plt.xlabel('Predikovaná třída')
plt.ylabel('Skutečná třída')
plt.tight_layout()
plt.show()

In [7]:
# Vizualizace 2: Důležitost příznaků (Feature Importances)
feat_importances = pd.Series(best_rf.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
feat_importances.plot(kind='barh', color='#10b981')
plt.title('Důležitost příznaků (Feature Importance – MDI)', fontsize=13)
plt.xlabel('Relativní důležitost')
plt.tight_layout()
plt.show()

### Shrnutí a interpretace výsledků:
- **Optimalizovaný model dosáhl na testovací sadě:**
  - **Accuracy:** 85.71 %
  - **Precision (třída 1 – nemoc srdce):** 85.71 % (36 ze 42 predikovaných nemocných skutečně nemoc má)
  - **Recall (třída 1 – nemoc srdce):** 83.72 % (ze 43 skutečně nemocných model zachytil 36)
  - **F1-Score:** 0.8471
- **Nejvýznamnější prediktory:** Počet zasažených cév (`ca`), talasémie (`thal_normal`, `thal_reversable`), maximální tepová frekvence (`maxhr`) a ST deprese (`oldpeak`).
- **Klinická poznámka:** Ačkoliv zadání požadovalo optimalizaci metriky **Precision**, v lékařské diagnostice je klíčové minimalizovat falešně negativní nálezy (FN = 7), aby nemocní pacienti neodcházeli bez diagnózy. V praxi se proto v kardiologii posouvá klasifikační práh tak, aby Recall dosahoval alespoň 90–95 %.